# ⏳ 05 — Shock Persistence by Shock Type (skeleton)

How long does a shock last, and does it depend on its type (market-wide crash, short squeeze, idiosyncratic
collapse)? Event-study cells moved here from the daily EDA and now run **in hours** on the Binance data.

1. Event windows around the catalogued crises (hourly, cumulative returns).
2. Persistence: hours until volatility is back to its pre-event range.
3. Persistence by shock type.
4. What the models should be told (persistence as a target or a feature).

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"
V2 = "../data/processed/v2"

from cryptograph.analysis.event_study import event_window

hp = torch.load(f"{V2}/1h/panel.pt", weights_only=False)
times = pd.to_datetime(hp["times"])
model = hp["model_coins"]
close = pd.read_parquet("../data/interim/hourly/master_1h.parquet")["close"].unstack("coin").reindex(index=times, columns=model)
r = np.log(close).diff()
events = pd.read_csv("../data/external/events.csv", parse_dates=["date"])
events = events[events["date"] >= "2020-01-08"]

## 1. Event windows (hourly)

In [2]:
rows = []
for _, e in events.iterrows():
    w = event_window(r, e["date"], pre=24, post=72, freq="h")
    rows.append(pd.DataFrame({"hour": w.index, "median cumulative return": w.median(axis=1).values,
                              "event": f"{e['date'].date()} {e['name'][:28]}"}))
ew = pd.concat(rows)
fig = px.line(ew, x="hour", y="median cumulative return", color="event",
              title="Median cumulative log return across the 30 coins, 24 h before to 72 h after each event (hour 0 = 00:00 UTC of the event date)")
fig.add_vline(x=0, line_dash="dash", line_color="#ff5252")
fig.update_layout(height=460, yaxis_tickformat=".0%")
fig.show()
display(ew[ew["hour"].isin([0, 6, 24, 72])].pivot(index="event", columns="hour", values="median cumulative return")
        .map("{:+.1%}".format))

hour,0,6,24,72
event,,,,
2020-03-12 COVID Black Thursday,-0.7%,-9.3%,-53.3%,-42.3%
2021-05-19 China bans financial institu,-0.5%,-13.6%,-51.5%,-46.2%
2022-05-09 Terra UST depeg / LUNA colla,+1.3%,-2.4%,-15.6%,-32.2%
2022-06-13 Celsius withdrawal freeze /,-3.3%,-7.5%,-12.2%,+0.8%
2022-11-08 FTX collapse,-0.4%,-6.2%,-15.2%,-16.6%
2023-03-10 SVB failure / USDC depeg,-1.1%,-1.6%,+1.3%,+9.5%
2024-01-10 US spot BTC ETF approval,+0.5%,-0.2%,+9.0%,+4.1%
2024-08-05 Yen carry-trade unwind,-4.6%,-16.2%,-3.3%,-5.3%
2025-10-10 Oct-10 liquidation cascade,+0.1%,+0.6%,-20.6%,-15.2%


*What this shows:* the median coin's path around each crisis, hour by hour. Event dates are calendar days
(hour 0 is midnight UTC), so some moves start before hour 0; the exact hour of each shock is located in the
training phase from the shock labels. This is the starting point for measuring persistence in hours.

## 2. Persistence in hours

> **To fill in the training phase.** This section stays empty until the models are trained (docs/model_plan.md).

## 3. Persistence by shock type

> **To fill in the training phase.** This section stays empty until the models are trained (docs/model_plan.md).

## 4. Implications for the models

> **To fill in the training phase.** This section stays empty until the models are trained (docs/model_plan.md).